# Documento de Análise — Inteligência de Sinistros Salutar Saúde

**Período de referência:** Jul/2024 – Jun/2025 (12 meses)  
**Base analisada:** 6.465 eventos de utilização | 1.316 beneficiários com sinistro | R$ 98,3 milhões em custo total  
**Catálogo/Schema:** `homologacao.salutar_saude`  
**Data de geração:** Julho/2025

---

## Sumário Executivo

Este documento responde a quatro perguntas de negócio sobre a carteira de saúde:
1. **Distribuição de custo** — Pareto e perfil dos high-cost claimants
2. **Segmentação** — Coortes com comportamento de utilização distinto
3. **Padrões de utilização** — Tipos de evento e especialidades que puxam o custo; sinais de uso evitável
4. **Recomendações acionáveis** — Onde o time comercial deve atuar

## 1. Distribuição de Custo e High-Cost Claimants

### Concentração tipo Pareto

| Fração de Beneficiários | % do Custo Total | Volume (R$) |
| --- | --- | --- |
| Top 5% (66 pessoas) | 27,9% | R$ 27,4M |
| Top 10% (132 pessoas) | 41,7% | R$ 41,1M |
| Top 20% (264 pessoas) | 61,7% | R$ 60,6M |

Apenas **132 beneficiários** (10% dos que tiveram sinistro) concentram R$ 41,1 milhões — um custo médio de **R$ 311.716/pessoa**, 6,5× acima da média geral (R$ 48.231).

### Perfil do High-Cost Claimant (Top 10% vs. Demais 90%)

| Característica | Top 10% | Demais 90% | Δ |
| --- | --- | --- | --- |
| Média de eventos/ano | 9,1 | 3,4 | +168% |
| Faixa 45–59 | 35,6% | 26,5% | +9pp |
| Faixa 60+ | 22,7% | 19,0% | +4pp |
| % com 45+ anos | **58,3%** | 45,5% | +13pp |
| % Titulares | 64,4% | 59,1% | +5pp |
| % Masculino | 43,2% | 39,9% | +3pp |
| % Custo em Cirurgias | **72,0%** | 58,3% | +14pp |
| Ticket Médio Cirurgia | R$ 88.719 | R$ 69.217 | +28% |

### Especialidades do grupo Top 10% (por custo)

| Especialidade | Eventos | Custo Total | Ticket Médio |
| --- | --- | --- | --- |
| Clínica Geral | 132 | R$ 5,4M | R$ 40.789 |
| Oncologia | 126 | R$ 4,6M | R$ 36.747 |
| Endocrinologia | 111 | R$ 4,3M | R$ 38.440 |
| Psiquiatria | 138 | R$ 4,1M | R$ 29.855 |
| Oftalmologia | 131 | R$ 4,0M | R$ 30.496 |
| Ginecologia | 115 | R$ 3,9M | R$ 33.879 |

> **Conclusão:** O high-cost claimant típico é titular, masculino, 45+ anos, com despesa dominada por cirurgias de alto ticket. Oncologia, Endocrinologia e Psiquiatria são as especialidades críticas.

In [0]:
%sql
-- PARETO: Concentração de custo por beneficiário (últimos 12 meses)
WITH custo_benef AS (
  SELECT
    u.beneficiario_id,
    SUM(u.valor_sinistro) AS custo_total
  FROM homologacao.salutar_saude.gold_fat_utilizacao u
  WHERE u.data_evento BETWEEN DATE '2024-07-01' AND DATE '2025-06-30'
  GROUP BY u.beneficiario_id
),
ranked AS (
  SELECT
    beneficiario_id,
    custo_total,
    ROW_NUMBER() OVER (ORDER BY custo_total DESC) AS rn,
    COUNT(*) OVER () AS total_benef,
    SUM(custo_total) OVER () AS custo_geral
  FROM custo_benef
),
pareto AS (
  SELECT
    rn, total_benef, custo_total, custo_geral,
    SUM(custo_total) OVER (ORDER BY rn) AS custo_acumulado,
    ROUND(rn * 100.0 / total_benef, 2) AS pct_beneficiarios,
    ROUND(SUM(custo_total) OVER (ORDER BY rn) * 100.0 / custo_geral, 2) AS pct_custo_acumulado
  FROM ranked
)
SELECT
  total_benef AS total_beneficiarios_com_sinistro,
  custo_geral AS custo_total_sinistros,
  MAX(CASE WHEN pct_beneficiarios <= 5.0 THEN pct_custo_acumulado END) AS pct_custo_top5pct,
  MAX(CASE WHEN pct_beneficiarios <= 10.0 THEN pct_custo_acumulado END) AS pct_custo_top10pct,
  MAX(CASE WHEN pct_beneficiarios <= 20.0 THEN pct_custo_acumulado END) AS pct_custo_top20pct
FROM pareto
GROUP BY total_benef, custo_geral

total_beneficiarios_com_sinistro,custo_total_sinistros,pct_custo_top5pct,pct_custo_top10pct,pct_custo_top20pct
1316,98251674.40,27.86,41.70,61.68


In [0]:
%sql
-- Perfil demográfico: Top 10% vs Demais 90%
WITH custo_benef AS (
  SELECT u.beneficiario_id, SUM(u.valor_sinistro) AS custo_total
  FROM homologacao.salutar_saude.gold_fat_utilizacao u
  WHERE u.data_evento BETWEEN DATE '2024-07-01' AND DATE '2025-06-30'
  GROUP BY u.beneficiario_id
),
ranked AS (
  SELECT *, NTILE(10) OVER (ORDER BY custo_total DESC) AS decil
  FROM custo_benef
)
SELECT
  CASE WHEN decil = 1 THEN 'Top 10%' ELSE 'Demais 90%' END AS grupo,
  COUNT(DISTINCT r.beneficiario_id) AS qtd_benef,
  ROUND(SUM(CASE WHEN b.sexo = 'Feminino' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_feminino,
  ROUND(SUM(CASE WHEN b.sexo = 'Masculino' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_masculino,
  ROUND(SUM(CASE WHEN b.faixa_etaria = '60+' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_60mais,
  ROUND(SUM(CASE WHEN b.faixa_etaria = '45-59' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_45_59,
  ROUND(SUM(CASE WHEN b.faixa_etaria = '30-44' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_30_44,
  ROUND(SUM(CASE WHEN b.tipo_beneficiario = 'Titular' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_titular
FROM ranked r
  JOIN homologacao.salutar_saude.gold_dim_beneficiario b ON r.beneficiario_id = b.beneficiario_id
GROUP BY CASE WHEN decil = 1 THEN 'Top 10%' ELSE 'Demais 90%' END
ORDER BY grupo

grupo,qtd_benef,pct_feminino,pct_masculino,pct_60mais,pct_45_59,pct_30_44,pct_titular
Demais 90%,1184,40.8,39.9,19.0,26.5,25.4,59.1
Top 10%,132,39.4,43.2,22.7,35.6,28.8,64.4


Databricks visualization. Run in Databricks to view.

## 2. Segmentação — Perfis de Utilização Distintos

### Por Setor + Porte (Top custo per-capita)

| Segmento | Custo/Capita | Freq/Capita | Observação |
| --- | --- | --- | --- |
| Logística / Média / Sem copay | R$ 150.798 | 5,9 | Maior custo per-capita da carteira |
| Serv. Financeiros / Grande / Sem copay | R$ 139.981 | 6,3 | Alta frequência |
| Indústria / Pequena / Com copay | R$ 99.347 | 5,0 | Copay não freou volume |
| Tecnologia / Média / Com copay | R$ 94.016 | 4,5 | Grande volume absoluto (R$ 4,9M) |

### Efeito da Coparticipação (Global)

| Métrica | Com Copay | Sem Copay | Δ |
| --- | --- | --- | --- |
| Custo per-capita | R$ 73.110 | R$ 77.322 | −5,4% |
| Frequência per-capita | 4,0 | 4,1 | −2,4% |
| % Eventos em PS | 14,6% | 13,8% | +0,8pp |

> **Insight:** A coparticipação atual gera redução marginal (−5%) no custo per-capita. O mecanismo não está sendo efetivo para modular comportamento, especialmente nas faixas 45+.

### Por Faixa Etária

| Faixa | Custo/Capita | Freq/Capita | % Custo Total |
| --- | --- | --- | --- |
| 60+ | R$ 94.670 | 5,3 | 24,4% |
| 45–59 | R$ 86.500 | 4,6 | 31,4% |
| 30–44 | R$ 73.728 | 3,8 | 25,5% |
| 18–29 | R$ 59.786 | 3,0 | 13,0% |
| 00–17 | R$ 38.600 | 2,6 | 5,8% |

> A faixa 45+ (45-59 e 60+) responde por **55,8% do custo** com apenas 34% dos beneficiários ativos.

In [0]:
%sql
-- Segmentação: custo per-capita por setor + porte + coparticipação
WITH util AS (
  SELECT u.empresa_id, u.plano_id, u.beneficiario_id, u.valor_sinistro
  FROM homologacao.salutar_saude.gold_fat_utilizacao u
  WHERE u.data_evento BETWEEN DATE '2024-07-01' AND DATE '2025-06-30'
)
SELECT
  e.setor, e.porte, p.coparticipacao,
  COUNT(DISTINCT u.beneficiario_id) AS benef_com_sinistro,
  COUNT(*) AS qtd_eventos,
  ROUND(SUM(u.valor_sinistro), 2) AS custo_total,
  ROUND(SUM(u.valor_sinistro) / COUNT(DISTINCT u.beneficiario_id), 2) AS custo_per_capita,
  ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT u.beneficiario_id), 1) AS freq_per_capita
FROM util u
  JOIN homologacao.salutar_saude.gold_dim_empresa e ON u.empresa_id = e.empresa_id
  JOIN homologacao.salutar_saude.gold_dim_plano p ON u.plano_id = p.plano_id
GROUP BY e.setor, e.porte, p.coparticipacao
ORDER BY custo_per_capita DESC

setor,porte,coparticipacao,benef_com_sinistro,qtd_eventos,custo_total,custo_per_capita,freq_per_capita
Logística,Média,Não,20,118,3015950.01,150797.50,5.9
Serviços Financeiros,Grande,Não,11,69,1539793.17,139981.20,6.3
Indústria,Pequena,Sim,23,115,2284979.05,99346.92,5.0
Indústria,Grande,Não,16,77,1554555.16,97159.70,4.8
Tecnologia,Média,Sim,52,233,4888810.14,94015.58,4.5
Agronegócio,Grande,Sim,61,310,5731676.06,93961.90,5.1
Varejo,Média,Sim,12,62,1104747.40,92062.28,5.2
Educação,Média,Não,23,92,1982580.03,86199.13,4.0
Saúde,Pequena,Não,122,553,10240810.31,83941.07,4.5
Serviços Financeiros,Pequena,Não,42,154,3523857.96,83901.38,3.7


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Segmentação por faixa etária e efeito da coparticipação
WITH util AS (
  SELECT u.beneficiario_id, u.plano_id, u.valor_sinistro
  FROM homologacao.salutar_saude.gold_fat_utilizacao u
  WHERE u.data_evento BETWEEN DATE '2024-07-01' AND DATE '2025-06-30'
)
SELECT
  b.faixa_etaria, p.coparticipacao,
  COUNT(DISTINCT u.beneficiario_id) AS benef_com_sinistro,
  COUNT(*) AS qtd_eventos,
  ROUND(SUM(u.valor_sinistro), 2) AS custo_total,
  ROUND(SUM(u.valor_sinistro) / COUNT(DISTINCT u.beneficiario_id), 2) AS custo_per_capita,
  ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT u.beneficiario_id), 1) AS freq_per_capita
FROM util u
  JOIN homologacao.salutar_saude.gold_dim_beneficiario b ON u.beneficiario_id = b.beneficiario_id
  JOIN homologacao.salutar_saude.gold_dim_plano p ON u.plano_id = p.plano_id
GROUP BY b.faixa_etaria, p.coparticipacao
ORDER BY b.faixa_etaria, p.coparticipacao

faixa_etaria,coparticipacao,benef_com_sinistro,qtd_eventos,custo_total,custo_per_capita,freq_per_capita
00-17,Não,60,166,1957494.07,32624.90,2.8
00-17,Sim,87,210,3715725.77,42709.49,2.4
18-29,Não,76,258,6502361.24,85557.38,3.4
18-29,Sim,138,386,6289470.43,45575.87,2.8
30-44,Não,123,440,8766229.84,71270.16,3.6
30-44,Sim,216,837,16243891.41,75203.20,3.9
45-59,Não,135,622,11677563.31,86500.47,4.6
45-59,Sim,226,971,19158137.27,84770.52,4.3
60+,Não,90,480,8520266.28,94669.63,5.3
60+,Sim,165,894,15420534.78,93457.79,5.4


Databricks visualization. Run in Databricks to view.

## 3. Padrões de Utilização

### Distribuição por Tipo de Evento

| Tipo | Qtd Eventos | % Eventos | % Custo | Ticket Médio |
| --- | --- | --- | --- | --- |
| Cirurgia | 815 | 15% | **64,0%** | R$ 77.209 |
| Internação | 776 | 15% | **32,6%** | R$ 41.240 |
| Exame de Imagem | 802 | 15% | 1,5% | R$ 1.862 |
| Pronto-Socorro | 753 | 14% | 1,1% | R$ 1.417 |
| Exame | 713 | 13% | 0,4% | R$ 597 |
| Consulta | 718 | 13% | 0,2% | R$ 256 |
| Terapia | 687 | 13% | 0,2% | R$ 226 |

> **Cirurgia + Internação = 96,6% do custo** com apenas 30% dos eventos. A gestão de custo passa obrigatoriamente pelo controle cirúrgico.

### Top Combinações Tipo × Especialidade (por custo)

| Tipo | Especialidade | Eventos | Custo | % Total |
| --- | --- | --- | --- | --- |
| Cirurgia | Clínica Geral | 103 | R$ 8,2M | 8,3% |
| Cirurgia | Oncologia | 97 | R$ 7,5M | 7,6% |
| Cirurgia | Dermatologia | 89 | R$ 6,8M | 6,9% |
| Cirurgia | Ortopedia | 87 | R$ 6,6M | 6,7% |
| Cirurgia | Psiquiatria | 78 | R$ 6,3M | 6,5% |
| Cirurgia | Cardiologia | 78 | R$ 6,3M | 6,4% |

### Uso Potencialmente Evitável — Pronto-Socorro

753 atendimentos de PS no período, distribuídos uniformemente entre especialidades (Oncologia 14%, Clínica Geral 11%, Pediatria 10%, Dermatologia 10%).

**Sinal de uso do PS como porta de entrada ambulatorial:**
- Ticket médio PS: R$ 1.417 — **5,5× o de uma consulta** (R$ 256)
- Distribuição homogênea entre especialidades sugere demanda não-urgente
- Empresas com >15% de eventos em PS: E31 (17,5%), R18 (18,1%), A01 (19,5%), M13 (17,3%)

> Redirecionando 40% desses PS para consulta ambulatorial = economia estimada de ~R$ 350K/ano + redução de internações subsequentes.

In [0]:
%sql
-- Padrões: custo por tipo de evento
SELECT
  tipo_evento,
  COUNT(*) AS qtd_eventos,
  ROUND(SUM(valor_sinistro), 2) AS custo_total,
  ROUND(AVG(valor_sinistro), 2) AS ticket_medio,
  ROUND(SUM(valor_sinistro) * 100.0 / (SELECT SUM(valor_sinistro) FROM homologacao.salutar_saude.gold_fat_utilizacao WHERE data_evento BETWEEN DATE '2024-07-01' AND DATE '2025-06-30'), 1) AS pct_custo_total
FROM homologacao.salutar_saude.gold_fat_utilizacao
WHERE data_evento BETWEEN DATE '2024-07-01' AND DATE '2025-06-30'
GROUP BY tipo_evento
ORDER BY custo_total DESC

tipo_evento,qtd_eventos,custo_total,ticket_medio,pct_custo_total
Cirurgia,815,62925593.88,77209.32,64.0
Internação,776,32002122.56,41239.85,32.6
Exame De Imagem,802,1493293.06,1861.96,1.5
Pronto-socorro,753,1066622.07,1416.50,1.1
Exame,713,425564.01,596.86,0.4
Consulta,718,183488.84,255.56,0.2
Terapia,687,154989.98,225.60,0.2


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Análise PS: uso evitável por especialidade
SELECT
  especialidade,
  COUNT(*) AS qtd_eventos,
  ROUND(SUM(valor_sinistro), 2) AS custo_total,
  ROUND(AVG(valor_sinistro), 2) AS ticket_medio_ps,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_do_total_ps
FROM homologacao.salutar_saude.gold_fat_utilizacao
WHERE data_evento BETWEEN DATE '2024-07-01' AND DATE '2025-06-30'
  AND tipo_evento = 'Pronto-socorro'
GROUP BY especialidade
ORDER BY qtd_eventos DESC

especialidade,qtd_eventos,custo_total,ticket_medio_ps,pct_do_total_ps
Oncologia,107,153261.27,1432.35,14.2
Clínica Geral,81,121604.05,1501.28,10.8
Pediatria,77,101921.49,1323.66,10.2
Dermatologia,76,96569.26,1270.65,10.1
Oftalmologia,74,113194.30,1529.65,9.8
Psiquiatria,74,98159.51,1326.48,9.8
Cardiologia,70,104890.36,1498.43,9.3
Ortopedia,66,99678.29,1510.28,8.8
Endocrinologia,64,87367.51,1365.12,8.5
Ginecologia,64,89976.03,1405.88,8.5


Databricks visualization. Run in Databricks to view.

## 4. Empresas Críticas — Sinistralidade Acima de 100%

21 empresas estão com sinistralidade acima de 80% nos últimos 12 meses. As 15 em **prejuízo técnico** (>100%):

| Empresa | Setor | Porte | Sinistralidade | Prejuízo Acum. |
| --- | --- | --- | --- | --- |
| Empresa K37 | Agronegócio | Grande | 1.218% | −R$ 1,3M |
| Empresa U21 | Saúde | Grande | 417% | −R$ 1,4M |
| Empresa G07 | Indústria | Pequena | 330% | −R$ 1,6M |
| Empresa I35 | Agronegócio | Pequena | 259% | −R$ 1,4M |
| Empresa N40 | Indústria | Grande | 247% | −R$ 851K |
| Empresa P16 | Educação | Média | 194% | −R$ 968K |
| Empresa T20 | Saúde | Grande | 194% | −R$ 2,0M |
| Empresa E31 | Serv. Financeiros | Pequena | 163% | −R$ 1,4M |
| Empresa D04 | Indústria | Média | 131% | −R$ 671K |
| Empresa E05 | Tecnologia | Média | 125% | −R$ 983K |
| Empresa F32 | Logística | Pequena | 118% | −R$ 536K |
| Empresa M13 | Educação | Grande | 114% | −R$ 407K |
| Empresa D30 | Serv. Financeiros | Grande | 113% | −R$ 642K |
| Empresa F06 | Varejo | Média | 110% | −R$ 166K |
| Empresa R18 | Serv. Financeiros | Média | 103% | −R$ 87K |

> **Prejuízo acumulado total das 15 empresas:** ~ R$ 13,9 milhões em 12 meses.

In [0]:
%sql
-- Empresas com sinistralidade > 80% nos últimos 12 meses
SELECT
  empresa_nome, setor, porte, uf,
  ROUND(SUM(receita_premios), 2) AS receita_12m,
  ROUND(SUM(custo_sinistros), 2) AS custo_12m,
  ROUND(SUM(custo_sinistros) * 100.0 / NULLIF(SUM(receita_premios), 0), 1) AS sinistralidade_12m,
  ROUND(SUM(margem_tecnica), 2) AS margem_12m,
  SUM(qtd_eventos) AS eventos_12m
FROM homologacao.salutar_saude.gold_fat_sinistralidade
WHERE ano_mes BETWEEN '2024-07' AND '2025-06'
GROUP BY empresa_nome, setor, porte, uf
HAVING SUM(custo_sinistros) * 100.0 / NULLIF(SUM(receita_premios), 0) > 80
ORDER BY sinistralidade_12m DESC

empresa_nome,setor,porte,uf,receita_12m,custo_12m,sinistralidade_12m,margem_12m,eventos_12m
Empresa K37 Ltda,Agronegócio,Grande,PR,117474.98,1431446.60,1218.5,-1313971.62,72
Empresa U21 Ltda,Saúde,Grande,BA,452325.16,1884787.43,416.7,-1432462.27,100
Empresa G07 Ltda,Indústria,Pequena,BA,692335.48,2284979.05,330.0,-1592643.57,115
Empresa I35 Ltda,Agronegócio,Pequena,CE,879283.98,2278703.39,259.2,-1399419.41,98
Empresa N40 Ltda,Indústria,Grande,BA,577021.44,1427862.60,247.5,-850841.16,62
Empresa P16 Ltda,Educação,Média,BA,1026028.06,1993877.71,194.3,-967849.65,95
Empresa T20 Ltda,Saúde,Grande,PR,2151657.38,4167539.59,193.7,-2015882.21,167
Empresa E31 Ltda,Serviços Financeiros,Pequena,PE,2160311.36,3523857.96,163.1,-1363546.60,154
Empresa D04 Ltda,Indústria,Média,MG,2186743.92,2858074.38,130.7,-671330.46,171
Empresa E05 Ltda,Tecnologia,Média,PR,3906138.31,4888810.14,125.2,-982671.83,233


Databricks visualization. Run in Databricks to view.

## 5. Recomendações Acionáveis (Priorizadas)

---

### Recomendação 1 — Programa de Gestão de Casos (Case Management) para Top 10%

| Item | Detalhe |
| --- | --- |
| **Alvo** | 132 beneficiários que concentram 42% do custo; foco inicial nos 66 do top 5% (28% do custo) |
| **Intervenção** | Gestor de caso individual; revisão de plano terapêutico com segunda opinião antes de cirurgias eletivas; navegação para ambulatório em vez de PS |
| **Impacto potencial** | Redução de 15% no custo do grupo = **R$ 6,2M/ano** |
| **KPI** | Custo per-capita trimestral do grupo monitorado vs. baseline; taxa de cirurgias com segunda opinião |

---

### Recomendação 2 — Abordagem Comercial nas 5 Empresas com Maior Prejuízo Absoluto

| Item | Detalhe |
| --- | --- |
| **Alvo** | T20 (−R$ 2,0M), G07 (−R$ 1,6M), U21 (−R$ 1,4M), K37 (−R$ 1,3M), E05 (−R$ 983K) |
| **Intervenção** | Reajuste técnico na renovação; migração para plano com coparticipação efetiva (co-pay por evento cirúrgico); programa de saúde corporativa (rastreamento preventivo para faixa 45+) |
| **Impacto potencial** | Recuperação de margem de **R$ 7,3M/ano** nos 5 contratos |
| **KPI** | Sinistralidade mensal pós-intervenção; taxa de adesão ao programa preventivo |

---

### Recomendação 3 — Redirecionamento de Pronto-Socorro para Ambulatório

| Item | Detalhe |
| --- | --- |
| **Alvo** | Empresas com >15% de eventos em PS (E31, R18, A01, M13 — ~595 eventos/ano) |
| **Intervenção** | Telemedicina 24h como primeira linha; orientação de rede (clínicas parceiras próximas à sede); copay diferenciado para PS sem encaminhamento |
| **Impacto potencial** | Redirecionando 40% dos PS para consulta = economia de **~R$ 350K/ano** + redução de internações subsequentes |
| **KPI** | Razão PS/Consulta por empresa (meta: <10%); NPS do beneficiário no canal digital |

---

### Recomendação 4 — Revisão do Modelo de Coparticipação

| Item | Detalhe |
| --- | --- |
| **Alvo** | Toda a carteira (832 beneficiários em planos com copay) |
| **Intervenção** | Copay atual é ineficaz (− 5% custo, 0% frequência). Redesenhar para copay progressivo: 30% em cirurgia eletiva, 0% em prevenção — incentivando cuidado ambulatorial precoce |
| **Impacto potencial** | Redução de 10% na frequência cirúrgica eletiva = **R$ 6,3M/ano** |
| **KPI** | Frequência por tipo de evento em planos reformulados vs. controle; satisfação do beneficiário |

---

## Fontes de Dados

| Tabela | Descrição |
| --- | --- |
| `homologacao.salutar_saude.gold_fat_utilizacao` | Fato de eventos/sinistros (grão: 1 evento) |
| `homologacao.salutar_saude.gold_fat_sinistralidade` | Fato de sinistralidade mensal por empresa |
| `homologacao.salutar_saude.gold_dim_beneficiario` | Dimensão de beneficiários (idade, sexo, tipo) |
| `homologacao.salutar_saude.gold_dim_empresa` | Dimensão de empresas (setor, porte, UF) |
| `homologacao.salutar_saude.gold_dim_plano` | Dimensão de planos (coparticipação, segmentação) |

**Notebook de análise exploratória:** `analise_comercial_gold`  
**Dashboard:** Análise Comercial — Salutar Saúde  
**Período:** Jul/2024 – Jun/2025